In [9]:
# Run from repo root: uv run jupyter notebook scripts/test_rrweb.ipynb
import os
import sys
import json

_cwd = os.getcwd()
_repo_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == "scripts" else _cwd
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
os.chdir(_repo_root)

from rrweb.normalizer import normalize_events, normalized_events_to_dict_list
from rrweb.compressor import compress_events, get_compression_stats
from rrweb.patterns import extract_patterns, patterns_to_text

# Legacy (kept for individual component testing)
from graphs.session_analysis.nodes.tier0_analyze import analyze_events

# Session Analysis Graph -- the new tiered entry point
from graphs.session_analysis import SessionAnalyzer

from settings import settings
os.environ["LANGFUSE_PUBLIC_KEY"] = settings.LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = settings.LANGFUSE_PRIVATE_KEY
os.environ["LANGFUSE_HOST"] = settings.LANGFUSE_HOST


In [10]:
EVENTS_PATH = os.path.join(_repo_root, "scripts", "assets", "events_2.json")
with open(EVENTS_PATH) as f:
    raw = json.load(f)

print("Session:", raw.get("sessionId"), "| Events in data:", len(raw.get("data", [])))
print("Start:", raw.get("startTime"), "End:", raw.get("endTime"))


Session: 4e157858-41e3-438f-8f89-159053e1a676 | Events in data: 164
Start: 1770839434494 End: 1770839467809


In [11]:
# Normalize: idempotent, LLM-friendly, token-efficient events
normalized = normalize_events(raw, collapse_scroll_ms=800)
print(f"Normalized events: {len(normalized)}\n")
for ev in normalized:
    print(f"  {ev.t:7.2f}s  {ev.e} {ev.node_id} {ev.node if ev.node else ''}")


Normalized events: 92

    17.58s  user moved mouse to (1000, 231) on element #203 None {'source': 1, 'positions': [{'x': 1000, 'y': 231, 'id': 203, 'timeOffset': 0}]}
    18.08s  user moved mouse to (607, 273) on element #246 None {'source': 1, 'positions': [{'x': 726, 'y': 258, 'id': 266, 'timeOffset': -450}, {'x': 629, 'y': 265, 'id': 244, 'timeOffset': -394}, {'x': 612, 'y': 269, 'id': 246, 'timeOffset': -338}, {'x': 607, 'y': 273, 'id': 246, 'timeOffset': -280}]}
    18.76s  user moved mouse to (607, 273) on element #246 None {'source': 1, 'positions': [{'x': 607, 'y': 273, 'id': 246, 'timeOffset': 0}]}
    19.26s  user moved mouse to (200, 142) on element #50 None {'source': 1, 'positions': [{'x': 577, 'y': 268, 'id': 246, 'timeOffset': -447}, {'x': 501, 'y': 251, 'id': 250, 'timeOffset': -391}, {'x': 374, 'y': 223, 'id': 203, 'timeOffset': -335}, {'x': 251, 'y': 180, 'id': 41, 'timeOffset': -283}, {'x': 204, 'y': 150, 'id': 50, 'timeOffset': -226}, {'x': 201, 'y': 143, 'id': 50,

## Tier 0


In [12]:
from rrweb.keyframes import get_keyframe_timestamps

events_list = normalized_events_to_dict_list(normalized)

ressss = get_keyframe_timestamps(events_list)
print(ressss)


[17.58, 20.9, 22.6, 25.8, 29.8, 31.8, 33.22, 33.22]


## Image-based AI analysis (keyframes + vision) + CPU & cost

Runs **analyze_with_images_ai**: reconstructs keyframes from rrweb, sends events + screenshots to a vision model. Use this to compare **wall-clock time**, **CPU usage**, and **estimated cost/tokens** vs text-only AI (cell above).

In [6]:
PRICING_PER_1M = {
    "gpt-4o": {"input": 2.50, "output": 10.0},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gemini-2.5-flash-preview-09-2025": {"input": 0.3, "output": 2.50},
}
def _estimate_cost(usage: dict, model: str) -> tuple[float, str]:
    """Returns (cost_usd, detail_str)."""
    if not usage:
        return 0.0, "no usage data"
    inp = usage.get("input_tokens") or usage.get("prompt_tokens") or 0
    out = usage.get("output_tokens") or 0
    prices = PRICING_PER_1M.get(model, PRICING_PER_1M["gpt-4o"])
    cost = (inp * prices["input"] + out * prices["output"]) / 1_000_000
    return cost, f"input={inp}, output={out}"
    
print(_estimate_cost({"input_tokens": 7000, "output_tokens": 2000}, "gemini-2.5-flash-preview-09-2025"))


(0.0071, 'input=7000, output=2000')


In [4]:
# prompt for later
UI_UX_PROMPT = """
Act as a **principal-level product designer and UX researcher** with strong AI-product experience. Think across **product strategy, UX, and UI**. I’m building a product that [add a quick summary about your product]
Below are 4 pillars about this product that you need to understand:

## 1. Problem & Customer
Define the product’s real problem, its importance, and why existing solutions are inadequate. Also, define the target audience and their specific needs for the product.

## 2. Core Value Proposition
State the main outcome the product delivers. What does the user achieve better or faster, and can this value be clearly expressed in a single, simple sentence?

## 3. Product Edge
Product should win due to a unique edge, compelling even with similar competitor capabilities.

## 4. Product Principles
Non-negotiable values for product and design decisions prioritize intentionality, guidance, human voice, engagement, and clarity.

### Your Task
Evaluate whether the **current UX and UI truly reflect these 4 pillars**, identify gaps or misalignment, and suggest concrete UX/UI improvements to strengthen focus, differentiation, and usability.
"""


In [13]:
# Test 1: Event Compression
print("=" * 60)
print("TEST 1: SEMANTIC EVENT COMPRESSION")
print("=" * 60)

# Get programmatic result first (needed for finding markers)
events_list = normalized_events_to_dict_list(normalized)
prog_result = analyze_events(events_list)

# Compress events
compressed = compress_events(events_list, prog_result)
stats = get_compression_stats(events_list, compressed)

print(f"\nCompression Statistics:")
print(f"  Original events: {stats['original_events']}")
print(f"  Original chars: {stats['original_chars']}")
print(f"  Compressed chars: {stats['compressed_chars']}")
print(f"  Compression ratio: {stats['compression_ratio']*100:.1f}%")
print(f"  Estimated tokens saved: {stats['estimated_tokens_saved']}")

print(f"\n--- Compressed Output ---")
print(compressed)


TEST 1: SEMANTIC EVENT COMPRESSION

Compression Statistics:
  Original events: 92
  Original chars: 5480
  Compressed chars: 863
  Compression ratio: 84.0%
  Estimated tokens saved: 1154

--- Compressed Output ---
  [!!] FORM_STRUGGLE: User struggled with element #-1 (7 inputs / corrections)
  [!!] FORM_STRUGGLE: User struggled with input "type:checkbox" (21 inputs / corrections)
[22s] custom [$url_changed]: (see payload)
[22s] Network: GET api/projects 200; GET api/users 200; GET api/users 200
[22s] Console [log]: Request {"url":"http://localhost:5174/api/users","method":"GET","headers":{"acce (+7 more)
[23s] Scrolled down (y=1→76)
[24s] → Navigated to projects
[25s] Scrolled (1 events)
[26s] Filled form: ✓ type:checkbox
[27-28s] Scrolled down (y=286→346)
  [!] DEAD_CLICK: Click on input "placeholder:Enter project name" had no visible effect
  [!] FORM_STRUGGLE: User struggled with textarea "placeholder:Enter project description" (3 inputs /
[30-33s] Filled form (15 fields): unchecked

In [14]:
# Test 2: Behavioral Pattern Extraction
from rrweb.patterns import BehavioralPattern


print("=" * 60)
print("TEST 2: BEHAVIORAL PATTERN EXTRACTION")
print("=" * 60)

patterns = extract_patterns(events_list)

print(f"\nDetected {len(patterns)} behavioral patterns:\n")

for i, p in enumerate[BehavioralPattern](patterns, 1):
    time_str = f"[{p.time_range[0]:.0f}s-{p.time_range[1]:.0f}s]" if p.time_range else ""
    severity_marker = "[!]" if p.severity == "warning" else "[-]"
    print(f"{i}. {severity_marker} {p.type}: {p.description} {time_str}")

print(f"\n--- Patterns as text for LLM ---")
print(patterns_to_text(patterns))


TEST 2: BEHAVIORAL PATTERN EXTRACTION

Detected 1 behavioral patterns:

1. [!] form_abandonment: User filled 33 form fields but never submitted [22s-33s]

--- Patterns as text for LLM ---
Behavioral Context:
[!] User filled 33 form fields but never submitted


In [15]:
# Session Analysis Graph — Tier 0 (Programmatic Only)
print("=" * 60)
print("SESSION ANALYSIS GRAPH — TIER 0 (Programmatic Only)")
print("=" * 60)

import time

from common.schemas.session_analysis import SessionAnalysisResult

analyzer = SessionAnalyzer()

start = time.time()
state_t0: SessionAnalysisResult = analyzer.analyze(raw, force_tier="tier0")
elapsed_t0 = time.time() - start

result_t0 = state_t0["result"]
print(f"\nTier: {state_t0['tier']}")
print(f"Time: {elapsed_t0:.2f}s")
print(f"Health Score: {result_t0.health_score}")
print(f"\nSummary: {result_t0.summary}")
print(f"\n--- Total Issues ({sum(len(i.issues) for i in result_t0.intervals)}) ---")
print(f"\n--- Intervals ---")
for i in result_t0.intervals:
    print(f"  [{i.short_title}] of [{i.start_time}-{i.end_time}] {i.key_events}")
    if i.issues:
        print(f"    Issues:")
        for f in i.issues:
            print(f"      [{f.severity.upper()}] {f.type}: {f.reproduction_steps}")
            if f.root_cause:
                print(f"        Root cause: {f.root_cause}")


Item exceeds size limit (size: 1516959), dropping input / output / metadata of item until it fits.


SESSION ANALYSIS GRAPH — TIER 0 (Programmatic Only)

Tier: tier0
Time: 0.28s
Health Score: 72.8

Summary: Session duration: 15s. User performed: 6 clicks, 1 navigations, 33 form inputs, 5 scrolls. Issues: 3 form struggle.

--- Total Issues (3) ---

--- Intervals ---
  [Session activity] of [00:17-00:21] ['[00:19] user click on element #42', '[00:21] user click on element #50']
    Issues:
      [HIGH] form_struggle: Attempt to fill element #-1 with corrections
        Root cause: User struggled with element #-1 (7 inputs / corrections)
  [Session activity] of [00:21-00:23] ['[22s] custom [$url_changed]: (see payload)', '[22s] Network: GET api/projects 200; GET api/users 200; GET api/users 200', '[22s] Console [log]: Request {"url":"http://localhost:5174/api/users","method":"GET","headers":{"acce (+7 more)']
    Issues:
      [HIGH] form_struggle: Attempt to fill input "type:checkbox" with corrections
        Root cause: User struggled with input "type:checkbox" (21 inputs / corrections

Item exceeds size limit (size: 1575986), dropping input / output / metadata of item until it fits.


In [7]:
# Session Analysis Graph — Tier 1 (Quick AI Summary)
print("=" * 60)
print("SESSION ANALYSIS GRAPH — TIER 1 (Quick AI Summary)")
print("=" * 60)

start = time.time()
state_t1 = analyzer.analyze(
    raw,
    force_tier="tier1",
    model="gemini-2.5-flash-preview-09-2025",
)
elapsed_t1 = time.time() - start

prog_result = state_t1["prog_result"]
print("PROG RESULT:", prog_result.intervals)
result_t1 = state_t1["result"]
print(f"\nTier: {state_t1['tier']}")
print(f"Time: {elapsed_t1:.2f}s")

print(f"\n--- AI Generated Summary ---")
print(result_t1.summary)

print(f"\n--- Total Issues ({sum(len(i.issues) for i in result_t1.intervals)}) ---")

print(f"\n--- Intervals ---")
for i in result_t1.intervals:
    print(f"  [{i.short_title}] {i.key_events}")
    if i.issues:
        print(f"    Issues:")
        for f in i.issues:
            print(f"      [{f.severity.upper()}] {f.type}: {f.reproduction_steps}")
            if f.root_cause:
                print(f"        Root cause: {f.root_cause}")


SESSION ANALYSIS GRAPH — TIER 1 (Quick AI Summary)


Item exceeds size limit (size: 26765274), dropping input / output / metadata of item until it fits.
Item exceeds size limit (size: 26809497), dropping input / output / metadata of item until it fits.


PROG RESULT: [TimestampInterval(start_time='00:00', end_time='00:03', short_title='Session activity', issues=[], key_events=['[00:01] user moved mouse to (655, 236) on element #464', '[00:00] user moved mouse to (655, 237) on element #464']), TimestampInterval(start_time='00:03', end_time='00:13', short_title='Navigation', issues=[], key_events=['[3s] → Navigated to #who-its-for', '[8-10s] Scrolled up (y=1924→533)', '[11s] → Clicked "Reserve Your Spot" button', '[11s] Scrolled down (y=377→4201)']), TimestampInterval(start_time='00:13', end_time='00:15', short_title='Navigation', issues=[], key_events=['[13s] → Navigated to #who-its-for']), TimestampInterval(start_time='00:15', end_time='00:23', short_title='Session activity', issues=[Issue(type='rage_click', frequency=3, timestamp='00:23', severity='medium', root_cause='User clicked p "An AI copilot that knows your stack inside out, " 3 times in 1.5s with no response', reproduction_steps='Click on p "An AI copilot that knows your stack

Item exceeds size limit (size: 26809607), dropping input / output / metadata of item until it fits.


In [ ]:
from graphs.session_analysis.nodes.reconcile import reconcile_node


In [8]:
# Session Analysis Graph — Tier 2 (Full + Images)
print("=" * 60)
print("SESSION ANALYSIS GRAPH — TIER 2 (Full AI + Images)")
print("=" * 60)

start = time.time()
try:
    state_t2 = analyzer.analyze(
        raw,
        force_tier="tier2",
        model="gemini-2.5-flash-preview-09-2025",
        max_frames=20,
    )
    elapsed_t2 = time.time() - start

    result_t2 = state_t2["result"]
    print(f"\nTier: {state_t2['tier']}")
    print(f"Time: {elapsed_t2:.2f}s")

    print(f"\n--- Title ---")
    print(result_t2.title)

    print(f"\n--- Summary ---")
    print(result_t2.summary)

    print(f"\n--- Issues ({sum(len(i.issues) for i in result_t2.intervals)}) ---")
    for interval in result_t2.intervals:
        print(f"  [{interval.short_title}] {interval.key_events}")
        for f in interval.issues:
            print(f"    [{f.severity.upper()}] {f.type}: {f.root_cause}")
            if f.reproduction_steps:
                print(f"      Reproduction: {f.reproduction_steps}")

except Exception as e:
    print("Tier 2 analysis failed:", e)
    raise


SESSION ANALYSIS GRAPH — TIER 2 (Full AI + Images)


Item exceeds size limit (size: 26765292), dropping input / output / metadata of item until it fits.
Item exceeds size limit (size: 26811429), dropping input / output / metadata of item until it fits.
Item exceeds size limit (size: 5461771), dropping input / output / metadata of item until it fits.



Tier: tier2
Time: 34.31s

--- Title ---
Lead form submission followed by critical dead click on main CTA

--- Summary ---
User successfully submitted the lead form but immediately became frustrated, rage-clicking the now-unresponsive "Reserve Your Spot" button 6 times. They then struggled with navigation, clicking dead elements and entering loops, indicating high post-conversion friction.

--- Issues (3) ---
  [Successful Lead Form Submission] ['Navigated to #who-its-for', 'Filled form: CTO, 1-10, UX Optimization, New feature planning', 'Clicked "Join" button (POST 200 success)', 'Saw "You\'re on the list" success message']
  [Post-Conversion Frustration and Dead Clicks] ['Rage-clicked "Reserve Your Spot" 6 times (dead click)', 'Clicked non-interactive text/divs multiple times', 'Entered navigation loop (visited # 5 times)', 'Clicked "Get Started" and "Contact Us" buttons']
    [CRITICAL] dead_click: After a successful form submission, the "Reserve Your Spot" button remains visible an

Item exceeds size limit (size: 26807897), dropping input / output / metadata of item until it fits.


In [ ]:
# Session Analysis Graph — Auto Tier (let the router decide)
print("=" * 60)
print("SESSION ANALYSIS GRAPH — AUTO TIER ROUTING")
print("=" * 60)

start = time.time()
state_auto = analyzer.analyze(raw, model="gemini-2.5-flash-preview-09-2025")
elapsed_auto = time.time() - start

result_auto = state_auto["result"]
print(f"\nAuto-selected Tier: {state_auto['tier']}")
print(f"Time: {elapsed_auto:.2f}s")
print(f"Health Score: {result_auto.health_score}")

print(f"\n--- Title ---")
print(result_auto.title)

print(f"\n--- Summary ---")
print(result_auto.summary)

print(f"\n--- Intervals ({len(result_auto.intervals)}) ---")
for i in result_auto.intervals:
    print(f"  [{i.start_time}-{i.end_time}] {i.short_title}")
    for f in i.issues:
        print(f"    [{f.severity.upper()}] {f.type}: {f.root_cause[:100]}")
